# Proyek 2M Sembako: Data Science & AI Logistics Preparation

## 1. Identifikasi Masalah UMKM & Peran Data
* **Masalah Bisnis:** Toko 2M Sembako menghadapi inefisiensi *unit economics* (biaya per pengiriman membengkak) pada pengiriman jarak $\ge$ 2,25 km dengan skema upah saat ini. Selain itu, asumsi kecepatan kurir konstan (20 km/jam) membuat estimasi waktu (*ETA*) sering meleset saat jam sibuk atau ketika kurir membawa muatan berat (*bulk*).
* **Peran Data:** Membangun dataset simulasi berbasis parameter logistik nyata untuk dianalisis, dibersihkan, dan diserahkan ke tim Gen AI guna melatih model regresi prediksi ETA.
## 2. Pertanyaan Bisnis
1. Faktor apa yang paling memengaruhi keterlambatan waktu tempuh pengiriman?
2. Pada titik jarak dan berat berapa 2M Sembako mulai mengalami kerugian operasional kurir?
3. Bagaimana cara membersihkan data mentah pengiriman agar siap digunakan oleh model Machine Learning?
## 3. Draf Ringkas README (Handover untuk Tim Gen AI)
- **Masalah:** Mengganti estimasi waktu statis dengan prediksi ETA dinamis berbasis Machine Learning.
- **Fitur/Target:** Memprediksi `waktu_tempuh_menit` berdasarkan `jarak_km`, `berat_kg`, dan status jam sibuk (`is_rush_hour`).
- **Preprocessing:** Pembersihan nilai anomali (jarak negatif, waktu mustahil) dan ekstraksi fitur tambahan.
## 4. Proses Data Wrangling (Gathering, Assessing, Cleaning, & Export)
Tahapan ini mensimulasikan pengumpulan data logistik, mendeteksi kotornya data, membersihkan anomali, dan mengekspor hasilnya menjadi `dataset_pengiriman_2m_clean.csv`.

Import Library

In [1]:
import pandas as pd
import numpy as np

1. GATHERING: Generate Simulated Data based on Draft_DS_Kurir logic

In [3]:
np.random.seed(42)
n_samples = 1500

jarak_km = np.random.choice([1.0, 2.5, 4.0, 5.5], size=n_samples, p=[0.5, 0.3, 0.15, 0.05]) + np.random.normal(0, 0.2, n_samples)
berat_kg = np.random.exponential(scale=8, size=n_samples)
jam_order = np.random.randint(8, 21, size=n_samples)

In [4]:
waktu_tempuh_menit = (jarak_km / 20 * 60) + 5

waktu_tempuh_menit += np.where(berat_kg > 15, np.random.randint(5, 15, n_samples), 0)
waktu_tempuh_menit += np.where((jam_order >= 16) & (jam_order <= 19), np.random.randint(10, 25, n_samples), 0)

In [5]:
jarak_km[::100] = -1.5
waktu_tempuh_menit[::150] = 500

df_raw = pd.DataFrame({
    'order_id': range(1, n_samples + 1),
    'jam_order': jam_order,
    'jarak_km': jarak_km,
    'berat_kg': berat_kg,
    'waktu_tempuh_menit': waktu_tempuh_menit
})

2. ASSESSING & CLEANING (Wrangling)

In [6]:
print("Shape sebelum cleaning:", df_raw.shape)

# Hapus jarak negatif dan waktu mustahil (Akurasi Data Checklist)
df_clean = df_raw[(df_raw['jarak_km'] > 0) & (df_raw['waktu_tempuh_menit'] > 0) & (df_raw['waktu_tempuh_menit'] < 180)].copy()

# Feature Engineering
df_clean['is_rush_hour'] = df_clean['jam_order'].apply(lambda x: 1 if 16 <= x <= 19 else 0)
df_clean['is_heavy'] = df_clean['berat_kg'].apply(lambda x: 1 if x >= 15 else 0)

Shape sebelum cleaning: (1500, 5)


In [7]:
print("Shape setelah cleaning:", df_clean.shape)

Shape setelah cleaning: (1480, 7)


3. EXPORT UNTUK TIM GEN AI

In [8]:
file_name = 'dataset_pengiriman_2m_clean.csv'
df_clean.to_csv(file_name, index=False)
print(f"Dataset berhasil disimpan sebagai {file_name}.")

Dataset berhasil disimpan sebagai dataset_pengiriman_2m_clean.csv.
